In [1]:
import os,re,json,gc,time,random,warnings,zipfile
from pathlib import Path
os.environ.update(HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1",TOKENIZERS_PARALLELISM="true");warnings.filterwarnings("ignore")
import numpy as np,pandas as pd,pyarrow.parquet as pq,torch
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from IPython.display import FileLink,display

B=Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
M=Path("/kaggle/input/models/f1aver/ecup-product-matching-bge-v3/pytorch/default/1")
O=Path("/kaggle/working/bge_v3_private_audit");O.mkdir(parents=True,exist_ok=True)
H,IH,L,I=B/"matches.parquet",B/"items_human.parquet",B/"matches_llm.parquet",B/"items.parquet"
SEED,MAXLEN,BS,N=20260825,320,128,500
assert torch.cuda.device_count()==2 and all(p.exists() for p in [H,IH,L,I,M/"model.safetensors"]),"Нужны 2xT4, датасеты и BGE V3"
np.random.seed(SEED);random.seed(SEED);T=time.time()
print("GPU:",[torch.cuda.get_device_name(i) for i in range(2)],"\nMODEL:",M,flush=True)

K=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
C2L=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi");L2C=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
UR=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
UM={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QR=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]

def text(n,a):
    p=[str(n) if n is not None else ""]
    try:d=json.loads(a) if isinstance(a,str) else {}
    except:d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v};pick=[];used=set()
        for w in K:
            for k,v in low.items():
                if w in k and k not in used:pick.append(f"{k}:{v}");used.add(k)
        p.append(" ; ".join(pick+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    s=" | ".join(p).replace("ё","е").replace("Ё","Е");z=[]
    for q in s.split():
        c=sum("\u0400"<=x<="\u04ff" for x in q);l=sum(x.isascii() and x.isalpha() for x in q)
        z.append(q.translate(C2L if l>=c else L2C) if c and l else q)
    s=re.sub(r"[×хХ](?=\d)","x"," ".join(z));u=set();e=[]
    for m in UR.finditer(s):un,k=UM[m.group(2).lower()];u.add(f"{float(m.group(1).replace(',','.'))*k:g}{un}")
    if u:e.append("ед: "+" ".join(sorted(u)[:12]))
    m=QR[2].search(s.lower());qty=int(m.group(1))*int(m.group(2)) if m else next((int(x.group(1)) for r in [QR[0],QR[1],QR[3]] if (x:=r.search(s.lower()))),None)
    if qty and 1<qty<=1000:e.append(f"кол-во: {qty}")
    return (s+(" | "+" | ".join(e) if e else ""))[:2000]

def comps(a,b):
    p={}
    def f(x):
        x=int(x);r=p.setdefault(x,x)
        while r!=p[r]:p[r]=p[p[r]];r=p[r]
        p[x]=r;return r
    for x,y in zip(a,b):
        x,y=f(x),f(y)
        if x!=y:p[y]=x
    return np.fromiter((f(x) for x in a),np.int64,len(a))

def split(df):
    df=df.copy();df["component"]=comps(df.id1,df.id2);rng=np.random.RandomState(SEED);mp={}
    for _,g in df.groupby("category",observed=True):
        u=np.array(sorted(g.component.unique()));rng.shuffle(u);nt=max(1,round(len(u)*.1));ne=max(1,round(len(u)*.1))
        mp.update({int(x):"tune" for x in u[:nt]});mp.update({int(x):"eval" for x in u[nt:nt+ne]});mp.update({int(x):"train" for x in u[nt+ne:]})
    df["split"]=df.component.map(mp);return df

def ph(a,b):
    a=np.asarray(a,np.uint64);b=np.asarray(b,np.uint64)
    return np.minimum(a,b)*np.uint64(11400714819323198485)^np.maximum(a,b)*np.uint64(14029467366897019727)

h=pd.read_parquet(H,columns=["id1","id2","target"])
hc=pd.read_parquet(IH,columns=["id","category"]).drop_duplicates("id").set_index("id").category
h["category"]=h.id1.map(hc).fillna("unknown").astype(str);h["target"]=(h.target>=.5).astype(np.int8);h=split(h)
a=h[h.split.isin(["tune","eval"])].reset_index(drop=True);print("human",h.split.value_counts().to_dict(),flush=True)

ll=pd.read_parquet(L,columns=["id1","id2","target"]);lc=comps(ll.id1,ll.id2);rng=np.random.RandomState(13);u=np.unique(lc)
v=set(u[rng.rand(len(u))<.03]);vm=np.fromiter((x in v for x in lc),bool,len(ll));conf=(ll.target.values<=.2)|(ll.target.values>=.8)
lh=ll[vm&conf].copy();lh["target"]=(lh.target>=.5).astype(np.int8);lh["component"]=lc[vm&conf]
trainhash=np.unique(ph(ll.id1.values[~vm],ll.id2.values[~vm]));a["llm_seen"]=np.isin(ph(a.id1,a.id2),trainhash)
overlap={s:float(a.loc[a.split==s,"llm_seen"].mean()) for s in ["tune","eval"]}
print("LLM holdout",len(lh),"overlap",overlap,flush=True);del ll,lc,vm,trainhash;gc.collect()

need=set(a.id1)|set(a.id2)|set(lh.id1)|set(lh.id2);tx={};cat={}
for path in [IH,I]:
    for batch in pq.ParquetFile(path).iter_batches(columns=["id","name","attributes","category"],batch_size=400000):
        d=batch.to_pandas();d=d[d.id.isin(need)]
        for i,n,x,c in d.itertuples(index=False,name=None):i=int(i);tx[i]=text(n,x);cat[i]=str(c);need.discard(i)
        if not need:break
    if not need:break
assert not need;lh["category"]=[cat[int(i)] for i in lh.id1];print("texts",len(tx),flush=True)

tok=AutoTokenizer.from_pretrained(M,local_files_only=True)
raw=AutoModelForSequenceClassification.from_pretrained(M,local_files_only=True,dtype=torch.float16).cuda().eval()
model=torch.nn.DataParallel(raw,[0,1]).eval()

@torch.inference_mode()
def pred(df,name):
    path=O/f"{name}.npz"
    if path.exists():z=np.load(path);return z["p"],z["gap"]
    order=np.argsort([len(tx[int(x)])+len(tx[int(y)]) for x,y in zip(df.id1,df.id2)])
    p=np.empty(len(df),np.float32);gap=np.empty(len(df),np.float32);t=time.time()
    for s in range(0,len(order),BS):
        ix=order[s:s+BS];x=[tx[int(df.id1.iloc[i])] for i in ix];y=[tx[int(df.id2.iloc[i])] for i in ix]
        ef=tok(x,y,padding=True,truncation=True,max_length=MAXLEN,pad_to_multiple_of=8,return_tensors="pt")
        er=tok(y,x,padding=True,truncation=True,max_length=MAXLEN,pad_to_multiple_of=8,return_tensors="pt")
        ef={k:v.cuda() for k,v in ef.items()};er={k:v.cuda() for k,v in er.items()}
        with torch.autocast("cuda",dtype=torch.float16):
            pf=torch.sigmoid(model(**ef,return_dict=False)[0].squeeze(-1).float())
            pr=torch.sigmoid(model(**er,return_dict=False)[0].squeeze(-1).float())
        pf,pr=pf.cpu().numpy(),pr.cpu().numpy();p[ix]=(pf+pr)/2;gap[ix]=abs(pf-pr)
        if s==0 or s//BS%100==0:print(name,s+len(ix),"/",len(df),f"{(s+len(ix))/max(1,time.time()-t):.0f}/s",flush=True)
    np.savez(path,p=p,gap=gap);return p,gap

pa,ga=pred(a,"human");pl,gl=pred(lh,"llm")

def macro(df,p,m=None,detail=False):
    if m is None:m=np.ones(len(df),bool)
    y=df.target.values;c=df.category.values
    r={z:average_precision_score(y[q],p[q]) for z in np.unique(c[m]) if (q:=m&(c==z)).sum() and len(np.unique(y[q]))==2}
    return (float(np.mean(list(r.values()))),r) if detail else float(np.mean(list(r.values())))

t=a.split.values=="tune";e=a.split.values=="eval";clean=~a.llm_seen.values
hm={"tune":macro(a,pa,t),"eval":macro(a,pa,e),"gap":macro(a,pa,e)-macro(a,pa,t),
    "clean_tune":macro(a,pa,t&clean),"clean_eval":macro(a,pa,e&clean),
    "clean_gap":macro(a,pa,e&clean)-macro(a,pa,t&clean)}

lr=np.random.RandomState(2026);u=np.unique(lh.component);q=set(u[lr.rand(len(u))<.5])
half=np.fromiter((x in q for x in lh.component),bool,len(lh))
lm={"full":macro(lh,pl),"half_a":macro(lh,pl,half),"half_b":macro(lh,pl,~half),
    "gap":macro(lh,pl,~half)-macro(lh,pl,half)}

_,ta=macro(a,pa,t,True);_,ea=macro(a,pa,e,True)
cats=pd.DataFrame([{"category":c,"tune_ap":ta.get(c,np.nan),"eval_ap":ea.get(c,np.nan),
                    "gap":ea.get(c,np.nan)-ta.get(c,np.nan),
                    "eval_pairs":int((e&(a.category.values==c)).sum()),
                    "positives":int(a.target.values[e&(a.category.values==c)].sum()),
                    "llm_overlap":float(a.loc[e&(a.category.values==c),"llm_seen"].mean())}
                   for c in sorted(set(ta)|set(ea))]).sort_values("gap")
cats.to_csv(O/"categories.csv",index=False)

ef=a[e].reset_index(drop=True);ep=pa[e];u=np.unique(ef.component);rng=np.random.RandomState(31415);runs=[]
while len(runs)<N:
    pubc=set(u[rng.rand(len(u))<.30]);pub=np.fromiter((x in pubc for x in ef.component),bool,len(ef))
    ps,pc=macro(ef,ep,pub,True);vs,vc=macro(ef,ep,~pub,True)
    if len(pc)==len(vc)==20:runs.append((ps,vs,vs-ps))

runs=pd.DataFrame(runs,columns=["public","private","delta"]);runs.to_csv(O/"pseudo_private.csv",index=False)
base=hm["eval"]
sim={"private_median":float(runs.private.median()),"private_p05":float(runs.private.quantile(.05)),
     "private_p10":float(runs.private.quantile(.10)),
     "p_drop_gt_.005":float((runs.private<base-.005).mean()),
     "p_drop_gt_.010":float((runs.private<base-.010).mean()),
     "p_private_.010_below_public":float((runs.delta<-.01).mean())}

risk=(2 if hm["clean_gap"]<-.01 else 1 if hm["clean_gap"]<-.003 else 0)
risk+=2 if sim["p_drop_gt_.010"]>.2 else 1 if sim["p_drop_gt_.010"]>.1 else 0
risk+=1 if abs(lm["gap"])>.01 else 0
risk+=1 if (cats.gap<-.02).sum()>=5 else 0
level="LOW" if risk<=1 else "MEDIUM" if risk<=3 else "HIGH"

report={"public_lb":.5535,"note":"Оценка риска, не настоящий private LB","risk":level,
        "risk_points":risk,"human":hm,"llm":lm,"overlap":overlap,
        "symmetry":{"human_mean":float(ga.mean()),"human_p95":float(np.quantile(ga,.95)),
                    "llm_mean":float(gl.mean()),"llm_p95":float(np.quantile(gl,.95))},
        "pseudo_private":sim,"fragile_categories":int((cats.gap<-.02).sum()),
        "minutes":(time.time()-T)/60}

(O/"report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
a.assign(predict=pa).to_parquet(O/"human_predictions.parquet",index=False)
print("\nCATEGORIES\n",cats.to_string(index=False),"\n\nFINAL REPORT\n",json.dumps(report,ensure_ascii=False,indent=2),flush=True)

Z=Path("/kaggle/working/bge_v3_private_audit.zip")
with zipfile.ZipFile(Z,"w",zipfile.ZIP_DEFLATED) as z:
    for p in O.glob("*"):
        if p.is_file():z.write(p,p.name)
print("SAVED",Z);display(FileLink(str(Z)))

GPU: ['Tesla T4', 'Tesla T4'] 
MODEL: /kaggle/input/models/f1aver/ecup-product-matching-bge-v3/pytorch/default/1
human {'train': 292428, 'eval': 36617, 'tune': 36609}
LLM holdout 191555 overlap {'tune': 0.0, 'eval': 0.0}
texts 395266


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

human 128 / 73226 65/s
human 12928 / 73226 139/s
human 25728 / 73226 110/s
human 38528 / 73226 94/s
human 51328 / 73226 86/s
human 64128 / 73226 82/s
llm 128 / 191555 128/s
llm 12928 / 191555 172/s
llm 25728 / 191555 145/s
llm 38528 / 191555 128/s
llm 51328 / 191555 116/s
llm 64128 / 191555 106/s
llm 76928 / 191555 98/s
llm 89728 / 191555 93/s
llm 102528 / 191555 89/s
llm 115328 / 191555 87/s
llm 128128 / 191555 84/s
llm 140928 / 191555 83/s
llm 153728 / 191555 81/s
llm 166528 / 191555 80/s
llm 179328 / 191555 79/s

CATEGORIES
                category  tune_ap  eval_ap       gap  eval_pairs  positives  llm_overlap
             Автотовары 0.723058 0.657688 -0.065370        1895        323          0.0
Галантерея и аксессуары 0.719896 0.685256 -0.034640        1789        294          0.0
                 Одежда 0.590801 0.557842 -0.032959        2327        256          0.0
      Ювелирные изделия 0.645132 0.612239 -0.032893        1884        226          0.0
      Красота и гигиена 0.

/kaggle/working/bge_v3_private_audit.zip